# ARTI 406 - Software Defect Prediction

Group 2 project notebook for predicting defective software modules using KC1 software metrics.


## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sns.set_theme(style="whitegrid")

## 2. Load Dataset

Place the KC1 CSV file at `data/raw/kc1.csv` before running this section.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "kc1.csv"

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Basic Dataset Check

In [ ]:
print(df.shape)
display(df.info())
display(df.describe().T)
print("Missing values:")
display(df.isna().sum())

## 4. Prepare Features and Target

In [ ]:
TARGET = "defects"

y = df[TARGET]
if y.dtype == bool:
    y = y.astype(int)
else:
    y = y.astype(str).str.lower().map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}).fillna(y).astype(int)

X = df.drop(columns=[TARGET]).select_dtypes(include=["number"])

print(X.shape)
display(y.value_counts().sort_index())

## 5. Train Baseline Models

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

models = {
    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42)),
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
}

rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"Model": name, **{metric: scores[f"test_{metric}"].mean() for metric in scoring}})

results = pd.DataFrame(rows)
results

## 6. Next Steps

- Add grid search for SVM and Random Forest.
- Plot confusion matrices.
- Plot ROC curves.
- Add Random Forest feature importance.
- Write the final comparison and conclusion.